<a href="https://colab.research.google.com/github/yusraanwar33-source/Deep-learning/blob/main/LSTM_Movie_Review_Sentiment_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Movie Review Sentiment Analyzer

### 1. Setup and Data Loading

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.20.0


In [2]:

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=5000)

print(f"Training sequences: {len(x_train)}")
print(f"Test sequences: {len(x_test)}")

print("\nFirst training review (word indices):\n", x_train[0])
print("\nFirst training review label (0=negative, 1=positive):", y_train[0])

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training sequences: 25000
Test sequences: 25000

First training review (word indices):
 [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 2, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 

### 2. Data Exploration

In [3]:

word_index = imdb.get_word_index()

reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(text):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text])

print("\nDecoded first training review:\n", decode_review(x_train[0]))

review_lengths = [len(x) for x in x_train]
print(f"\nAverage review length: {np.mean(review_lengths):.2f} words")
print(f"Minimum review length: {np.min(review_lengths)} words")
print(f"Maximum review length: {np.max(review_lengths)} words")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Decoded first training review:
 ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly ? was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little ? that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big ? for the whole film but these children are a

### 3. Data Preprocessing: Padding Sequences

In [5]:

maxlen = 500

x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(f"Shape of x_train after padding: {x_train.shape}")
print(f"Shape of x_test after padding: {x_test.shape}")

print("\nFirst training review after padding (first 20 elements):\n", x_train[0][:20])
print("\nLast 20 elements of the first training review after padding:\n", x_train[0][-20:])

Shape of x_train after padding: (25000, 500)
Shape of x_test after padding: (25000, 500)

First training review after padding (first 20 elements):
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Last 20 elements of the first training review after padding:
 [  65   16   38 1334   88   12   16  283    5   16 4472  113  103   32
   15   16    2   19  178   32]


### 3. Data Preprocessing: Padding Sequences

In [4]:

maxlen = 500

x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(f"Shape of x_train after padding: {x_train.shape}")
print(f"Shape of x_test after padding: {x_test.shape}")
print("\nFirst training review after padding (first 20 elements):\n", x_train[0][:20])
print("\nLast 20 elements of the first training review after padding:\n", x_train[0][-20:])

Shape of x_train after padding: (25000, 500)
Shape of x_test after padding: (25000, 500)

First training review after padding (first 20 elements):
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Last 20 elements of the first training review after padding:
 [  65   16   38 1334   88   12   16  283    5   16 4472  113  103   32
   15   16    2   19  178   32]


### 4. Build the LSTM Model

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

vocabulary_size = 5000
embedding_dim = 128

model = Sequential([
    Embedding(vocabulary_size, embedding_dim, input_shape=(maxlen,)),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

### 5. Train the Model

In [ ]:
epochs = 10
batch_size = 32

print("Training the model...")
history = model.fit(x_train, y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.2)

print("Model training complete.")

Training the model...
Epoch 1/10
272/625 ━━━━━━━━━━━━━━━━━━━━ 4:36 782ms/step - accuracy: 0.8360 - loss: 0.3795

### 6. Evaluate the Model

In [ ]:
print("\nEvaluating the model...")
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

#prediction

In [ ]:
from tensorflow.keras.preprocessing import sequence

def predict_sentiment(review_text):
    # Tokenize the input review
    # Convert words to integers using the word_index
    word_to_int = []
    for word in review_text.lower().split():
        # Get the integer ID for the word, default to 0 if not found
        # Shift by 3 because 0, 1, 2 are reserved in IMDB dataset
        word_to_int.append(word_index.get(word, 0) + 3)

    # Pad the sequence to the same length as training data
    # The sequence.pad_sequences function expects a list of sequences
    padded_review = sequence.pad_sequences([word_to_int], maxlen=maxlen)

    # Make prediction
    prediction = model.predict(padded_review)

    # Interpret prediction
    # The output is a probability between 0 and 1
    if prediction[0][0] >= 0.5:
        sentiment = "Positive"
        probability = prediction[0][0]
    else:
        sentiment = "Negative"
        probability = 1 - prediction[0][0]

    print(f"\nReview: {review_text}")
    print(f"Predicted Sentiment: {sentiment} (Probability: {probability:.2f})")

# Example usage:
new_review_positive = "This movie was absolutely fantastic! I loved every minute of it. A must-watch."
predict_sentiment(new_review_positive)

new_review_negative = "Terrible acting and boring plot. I regret watching this film, complete waste of time."
predict_sentiment(new_review_negative)

new_review_neutral = "The movie was okay, not great, not bad. It had some interesting parts but nothing spectacular."
predict_sentiment(new_review_neutral)
